In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.svm import SVR
import pickle

# Load the dataset
df = pd.read_csv(r"C:\Users\Raj\SoilStream\cloud_functions\ml_model\mumbai_ratnagiri_weather_complete_1995_2025(N) - Binary.csv")

# Convert time to datetime
df['time'] = pd.to_datetime(df['time'])

# Separate data for Mumbai and Ratnagiri
df_mum = df[df['station'] == 'Mumbai'][['time', 'temp', 'hum', 'pres', 'binary_prcp']]
df_rat = df[df['station'] == 'Ratnagiri'][['time', 'temp', 'hum', 'pres', 'binary_prcp']]

# Merge on time to align dates
df_merged = pd.merge(df_mum, df_rat, on='time', suffixes=('_mum', '_rat'))

# Create regional target: 1 if rain in either station, else 0
df_merged['binary_prcp_regional'] = ((df_merged['binary_prcp_mum'] == 1) | (df_merged['binary_prcp_rat'] == 1)).astype(int)

# Create dataset where each station's features are used separately with the same regional target
df_mum_features = df_merged[['time', 'temp_mum', 'hum_mum', 'pres_mum', 'binary_prcp_regional']]
df_mum_features.columns = ['time', 'temp', 'hum', 'pres', 'binary_prcp']
df_rat_features = df_merged[['time', 'temp_rat', 'hum_rat', 'pres_rat', 'binary_prcp_regional']]
df_rat_features.columns = ['time', 'temp', 'hum', 'pres', 'binary_prcp']

# Concatenate to augment the dataset
df_konkan = pd.concat([df_mum_features, df_rat_features], ignore_index=True)

# Sort by time
df_konkan = df_konkan.sort_values('time')

# Create lagged features (7 days of past temp, hum, pres) to capture temporal dependencies
def create_lagged_features(df, features_cols, lag=7):
    lagged_df = df.copy()
    for feature in features_cols:
        for i in range(1, lag + 1):
            lagged_df[f'{feature}_lag{i}'] = lagged_df[feature].shift(i)
    return lagged_df

# Apply lagged features
features_cols = ['temp', 'hum', 'pres']
df_konkan = create_lagged_features(df_konkan, features_cols, lag=7)

# Add seasonal feature (month) to capture monsoon patterns
df_konkan['month'] = df_konkan['time'].dt.month

# Drop rows with NaN values (due to lagging)
df_konkan = df_konkan.dropna()

# Select features and target
features = [f'{feat}_lag{i}' for feat in features_cols for i in range(1, 8)] + features_cols + ['month']
X = df_konkan[features]
y = df_konkan['binary_prcp']

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split into train and test (80/20, chronological)
split_idx = int(0.8 * len(X_scaled))
X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Initialize and train SVR model (adapted for classification)
svr_model = SVR(kernel='rbf', C=1.0)
svr_model.fit(X_train, y_train)

# Make predictions with SVR
y_pred_prob_svr = svr_model.predict(X_test)
y_pred_svr = (y_pred_prob_svr >= 0.5).astype(int)

# Calculate metrics for SVR
print("SVR Metrics:")
accuracy_svr = accuracy_score(y_test, y_pred_svr)
precision_svr = precision_score(y_test, y_pred_svr)
recall_svr = recall_score(y_test, y_pred_svr)
f1_svr = f1_score(y_test, y_pred_svr)
print(f'Accuracy: {accuracy_svr:.4f}')
print(f'Precision: {precision_svr:.4f}')
print(f'Recall: {recall_svr:.4f}')
print(f'F1-Score: {f1_svr:.4f}')

# Save the model and scaler
with open('konkan_rainfall_svr_model.pkl', 'wb') as f:
    pickle.dump(svr_model, f)
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)